# Study Evaluation I

In [3]:
import numpy as np               # this import is not really necessary because we use it via pandas
import pandas as pd              # to work with our data
import matplotlib.pyplot as plt  # to plot charts
import plotly.graph_objects as go
import plotly.express as px
import plotly.io as pio
import seaborn as sns            # to make our charts prettier

from plotly.subplots import make_subplots
from sklearn import preprocessing

data_folder = '../data/questionnaires/'

sns.set(style="darkgrid")

In [5]:
# Load general participant metadata from the questionnaire export.
df_allgemein = pd.read_csv(rf'{data_folder}Allgemeine_Angaben.tsv', sep='\t')

# Rename verbose survey questions to concise analysis-friendly column names.
df_allgemein.rename(columns={"Zeitstempel": "Timestamp", 
                            "Probanden-ID": "PID",
                            "Wie alt sind Sie?": "Alter",
                            "Welchem Geschlecht fühlen Sie sich zugehörig?": "Geschlecht",
                            "Wie groß sind Sie?": "Größe",
                            "Was ist Ihr höchster Bildungsabschluss?": "Abschluss",
                            "Welche Tätigkeit üben Sie derzeit aus?": "Tätigkeit",
                            "In welcher Branche sind Sie tätig?": "Branche",
                            "Wie würden Sie Ihre Vorerfahrung mit Elastic Displays einschätzen?": "Vorerfahrung"},
                    inplace=True)

## Participants

In [6]:
# Column overview for all participants

df_allgemein

,Timestamp,PID,Alter,Geschlecht,Größe,Abschluss,Tätigkeit,Branche,Vorerfahrung
0,10.05.2021 14:06:21,1,18,männlich,178,Abitur,Student,keine,1
1,11.05.2021 14:19:35,2,27,männlich,175,Studium (Uni/FH),Student,Informatik,6
2,11.05.2021 16:25:40,3,23,weiblich,173,Abitur,Student,Werkstudenten in MMS Dresden,1
3,19.05.2021 14:09:50,4,29,weiblich,163,Studium (Uni/FH),Wissenschaftliche Mitarbeiterin,Hochschule,1
4,21.05.2021 09:31:42,5,28,männlich,183,Studium (Uni/FH),"wissenschaftlicher Mitarbeiter, Informatik",Softwareentwicklung,1
5,21.05.2021 11:21:40,6,25,männlich,183,Studium (Uni/FH),Wissenschaftlicher Mitarbeiter,öffentlicher Dienst,1
6,21.05.2021 14:39:42,7,28,männlich,180,Studium (Uni/FH),Student,Informatik,3
7,25.05.2021 14:23:19,8,26,männlich,160,Studium (Uni/FH),Student - Master,Informatik,3
8,25.05.2021 16:29:29,9,20,weiblich,178,Abitur,Studium,Medieninformatik,1
9,27.05.2021 11:26:06,10,19,männlich,182,Abitur,"Studieren (2. Sem), Business (privat)",Informatik/Design,3


In [8]:
correlations = df_allgemein.iloc[:,2:].corr(method='kendall', min_periods=1)
correlations

ValueError: could not convert string to float: 'männlich'

In [5]:
correlations = df_allgemein.iloc[:,2:].corr(method='spearman', min_periods=1)
correlations

,Alter,Größe,Abschluss_rank,Vorerfahrung
Alter,1.000000,0.125766,0.858094,0.023978
Größe,0.125766,1.000000,0.295091,-0.125054
Abschluss_rank,0.858094,0.295091,1.000000,0.152528
Vorerfahrung,0.023978,-0.125054,0.152528,1.000000


In [9]:
# Quick statistical summary

df_allgemein.describe().astype(int)

,PID,Alter,Größe,Vorerfahrung
count,24,24,24,24
mean,12,27,174,1
std,7,7,8,1
min,1,18,155,1
25%,6,23,171,1
50%,12,27,175,1
75%,18,30,180,3
max,24,47,192,6


In [10]:
data = df_allgemein[['Alter','Geschlecht']]
gtv_colormap = {1: 'rgb(54, 161, 185)', 2: 'rgb(0, 44, 61)', 3: 'rgb(233, 71, 74)'}

fig = go.Figure()

# male
fig.add_trace(go.Violin(y=data['Alter'][ data['Geschlecht'] == 'männlich' ],
                        scalegroup='alter', name='male',
                        line_color='rgb(233, 71, 74)')
             )
# all
fig.add_trace(go.Violin(y=data['Alter'],
                        scalegroup='alter', name='all',
                        line_color='rgb(0, 44, 61)')
             )

# female
fig.add_trace(go.Violin(y=data['Alter'][ data['Geschlecht'] == 'weiblich' ],
                        scalegroup='alter', name='female',
                        line_color='rgb(233, 71, 74)')
             )

fig.update_traces(meanline_visible=True,
                  points='all', # show all points
                  jitter=0.5,  # for better visibility
                  scalemode='count') # scale with total count

fig.update_yaxes(title="age")

fig.update_layout(
    #title_text="<b>Age</b>",
    showlegend=False,
    violingap=0, violingroupgap=0.4, violinmode='overlay',
    plot_bgcolor = 'rgba(239, 240, 231,.7)'
)

fig.show()

**Key Facts**
- 24 participants (13 male / 11 female) (N=24)
- Average age: 27 years (M=27, SD=7)
- Average height: 1.74 m (M=174, SD=8)
- Mostly no prior experience (M=1, SD=1)
- Education: 8 high school diplomas, 15 university degrees (UAS), 1 doctorate
- Only one clearly positive correlation: age and education level (Kendall's Tau=0.738)

## NASA RTLX

The **NASA TLX** questionnaire measures a person's perceived subjective workload during a specific activity or test task. It focuses on mental, physical, and temporal workload while also capturing performance, effort, and frustration. The **NASA RTLX** (Raw Task Load Index, used here) is a form of NASA TLX in which ratings are averaged and summed to estimate total workload. The user study analysis is reported separately by blocks (1, 2, 3).

In [78]:
df_nasa_rtlx = pd.read_csv(rf'{data_folder}NASA_RTLX_complete.tsv', sep='\t') # load all NASA RTLX data
df_nasa_rtlx

,PID,Block,mental effort,physical effort,temporal effort,performance,effort,frustration
0,1,1,25,40,60,75,50,30
1,1,2,50,30,40,75,50,15
2,1,3,15,30,15,70,40,10
3,2,1,10,50,25,80,50,15
4,2,2,15,45,35,75,50,35
...,...,...,...,...,...,...,...,...
64,23,2,10,20,0,60,20,50
65,23,3,10,20,10,20,20,10
66,24,1,60,50,50,60,65,55
67,24,2,65,50,35,65,65,40


In [79]:
# remove PID 10 due to unvalid data
#df_nasa_rtlx = df_nasa_rtlx[df_nasa_rtlx['PID'] != 10]
df_nasa_rtlx

,PID,Block,mental effort,physical effort,temporal effort,performance,effort,frustration
0,1,1,25,40,60,75,50,30
1,1,2,50,30,40,75,50,15
2,1,3,15,30,15,70,40,10
3,2,1,10,50,25,80,50,15
4,2,2,15,45,35,75,50,35
...,...,...,...,...,...,...,...,...
64,23,2,10,20,0,60,20,50
65,23,3,10,20,10,20,20,10
66,24,1,60,50,50,60,65,55
67,24,2,65,50,35,65,65,40


In [80]:
# new column: calculate unweighted NASA-TLX workload (per participant, per block)
df_nasa_rtlx['overall workload'] = df_nasa_rtlx.iloc[:,-6:].sum(axis=1) / 6

# split data blockwise (1,2,3)
df_nasa_rtlx_1 = df_nasa_rtlx.loc[df_nasa_rtlx['Block'] == 1]
df_nasa_rtlx_2 = df_nasa_rtlx.loc[df_nasa_rtlx['Block'] == 2]
df_nasa_rtlx_3 = df_nasa_rtlx.loc[df_nasa_rtlx['Block'] == 3]

df_nasa_rtlx.head()

,PID,Block,mental effort,physical effort,temporal effort,performance,effort,frustration,overall workload
0,1,1,25,40,60,75,50,30,46.666667
1,1,2,50,30,40,75,50,15,43.333333
2,1,3,15,30,15,70,40,10,30.000000
3,2,1,10,50,25,80,50,15,38.333333
4,2,2,15,45,35,75,50,35,42.500000


In [81]:
df_nasa_rtlx.describe()

,PID,Block,mental effort,physical effort,temporal effort,performance,effort,frustration,overall workload
count,69.000000,69.000000,69.000000,69.000000,69.000000,69.000000,69.000000,69.000000,69.000000
mean,12.608696,2.000000,37.391304,42.463768,30.579710,43.985507,49.130435,47.826087,41.896135
std,7.102645,0.822478,24.443944,23.493165,21.100882,23.366953,21.945060,26.282938,15.914321
min,1.000000,1.000000,5.000000,0.000000,0.000000,0.000000,5.000000,0.000000,8.333333
25%,6.000000,1.000000,15.000000,20.000000,20.000000,25.000000,30.000000,25.000000,30.000000
50%,13.000000,2.000000,30.000000,40.000000,25.000000,45.000000,50.000000,50.000000,42.500000
75%,19.000000,3.000000,60.000000,60.000000,45.000000,60.000000,70.000000,70.000000,50.000000
max,24.000000,3.000000,90.000000,90.000000,80.000000,85.000000,90.000000,95.000000,76.666667


In [82]:
df_nasa_rtlx_1 .describe()

,PID,Block,mental effort,physical effort,temporal effort,performance,effort,frustration,overall workload
count,23.000000,23.0,23.000000,23.000000,23.000000,23.000000,23.000000,23.000000,23.000000
mean,12.608696,1.0,40.000000,41.521739,32.173913,45.869565,51.521739,48.913043,43.333333
std,7.209458,0.0,26.068093,20.639480,23.297780,23.676425,20.194215,27.918572,13.775802
min,1.000000,1.0,5.000000,0.000000,0.000000,0.000000,15.000000,0.000000,19.166667
25%,6.500000,1.0,17.500000,22.500000,17.500000,32.500000,35.000000,25.000000,34.166667
50%,13.000000,1.0,35.000000,40.000000,25.000000,45.000000,50.000000,50.000000,41.666667
75%,18.500000,1.0,65.000000,60.000000,50.000000,60.000000,67.500000,72.500000,49.583333
max,24.000000,1.0,85.000000,75.000000,80.000000,85.000000,90.000000,95.000000,73.333333


### Overall Overview

In [83]:
df_nasa_rtlx.describe() # compact statistical overview of all NASA RTLX data

,PID,Block,mental effort,physical effort,temporal effort,performance,effort,frustration,overall workload
count,69.000000,69.000000,69.000000,69.000000,69.000000,69.000000,69.000000,69.000000,69.000000
mean,12.608696,2.000000,37.391304,42.463768,30.579710,43.985507,49.130435,47.826087,41.896135
std,7.102645,0.822478,24.443944,23.493165,21.100882,23.366953,21.945060,26.282938,15.914321
min,1.000000,1.000000,5.000000,0.000000,0.000000,0.000000,5.000000,0.000000,8.333333
25%,6.000000,1.000000,15.000000,20.000000,20.000000,25.000000,30.000000,25.000000,30.000000
50%,13.000000,2.000000,30.000000,40.000000,25.000000,45.000000,50.000000,50.000000,42.500000
75%,19.000000,3.000000,60.000000,60.000000,45.000000,60.000000,70.000000,70.000000,50.000000
max,24.000000,3.000000,90.000000,90.000000,80.000000,85.000000,90.000000,95.000000,76.666667


In [84]:
mean1 = df_nasa_rtlx_1.iloc[:,-7:-1].stack().dropna().mean()
mean1

43.333333333333336

In [85]:
mean2 = df_nasa_rtlx_2.iloc[:,-7:-1].stack().dropna().mean()
mean2

41.44927536231884

In [86]:
mean3 = df_nasa_rtlx_3.iloc[:,-7:-1].stack().dropna().mean()
mean3

40.905797101449274

In [87]:
mean_all = df_nasa_rtlx.iloc[:,-7:-1].stack().dropna().mean()
mean_all

41.89613526570049

In [88]:
# Box plots in Plotly Express, see: https://plotly.github.io/plotly.py-docs/generated/plotly.express.box.html
# See color_discrete_map for custom colors via a color dictionary

gtv_colormap = {1: 'rgb(54, 161, 185)', 2: 'rgb(0, 44, 61)', 3: 'rgb(233, 71, 74)'}
                         
# Use columns: Block, the six NASA-TLX dimensions, and overall workload
fig = px.box(df_nasa_rtlx.iloc[:,-8:], points=False, color='Block', color_discrete_map=gtv_colormap)

fig.add_vline(x=5.5, line_color='#fff', line_width=10)
fig.update_xaxes(title=None)
fig.update_yaxes(title=None)
fig.update_layout(    
    font_family='Roboto Condensed',
    font_size=15,
    legend=dict(
        yanchor="top",
        y=1.0,
        xanchor="left",
        x=0.95
    ),
    margin=go.layout.Margin(
        l=0, #left margin
        r=0, #right margin
        b=0, #bottom margin
        t=0, #top margin
    ),
    plot_bgcolor = 'rgba(239, 240, 231,.7)'
)

pio.write_image(fig, 'NASA_TLX_all_blockwise.pdf')
fig.show()

In [ ]:
# Box plots in Plotly Express, see: https://plotly.github.io/plotly.py-docs/generated/plotly.express.box.html
# See color_discrete_map for custom colors via a color dictionary
                        
# Use columns of the six NASA-TLX dimensions and overall workload
fig = px.box(df_nasa_rtlx.iloc[:,-7:], points=False)

fig.add_vline(x=5.5, line_color='#fff', line_width=10)
fig.add_hline(y=mean_all, 
              line_color='#e7267a', 
              annotation_text='<span style="color: rgb(233, 71, 74);">M=<br>'+ '{:.2f}'.format(mean_all) + '<br><br>SD=<br>15.91<span>',
              annotation_position='right',
              annotation_align='left')
             
fig.update_xaxes(title=None)#title="NASA-TLX dimension")
fig.update_yaxes(title=None)
fig.update_layout(    
    font_family='Roboto Condensed',
    font_size=15,
    legend=dict(
        yanchor="top",
        y=1.0,
        xanchor="left",
        x=0.95
    ),
    margin=go.layout.Margin(
        l=0, #left margin
        r=35, #right margin
        b=0, #bottom margin
        t=0, #top margin
    ),
    plot_bgcolor = 'rgba(239, 240, 231,.7)'
)
fig.update_traces(marker_color='#47626E')

pio.write_image(fig, 'NASA_TLX_all_mean.pdf')
fig.show()

In [90]:
# OVERALL WORKLOAD PER BLOCK

# Recompute workload from the six NASA-TLX dimensions for this standalone chart.
data = df_nasa_rtlx.iloc[:,1:8]
data['workload'] = data.iloc[:,-6:].sum(axis=1) / 6

# Split the derived workload values by study block.
b1 = data[data['Block'] == 1]
b2 = data[data['Block'] == 2]
b3 = data[data['Block'] == 3]
mean = data['workload'].mean()

# box plots
fig = go.Figure()
fig.add_trace(go.Box(
    y=b1['workload'],
    x=b1['Block'],
    name='Block 1',
    marker_color='rgba(54, 161, 185,1)',
    fillcolor='rgba(54, 161, 185,.5)'
))

fig.add_trace(go.Box(
    y=b2['workload'],
    x=b2['Block'],
    name='Block 2',
    marker_color='rgba(0, 44, 61,.8)',
    fillcolor = 'rgba(0, 44, 61,.4)'
))

fig.add_trace(go.Box(
    y=b3['workload'],
    x=b3['Block'],
    name='Block 3',
    marker_color='rgba(233, 71, 74,.8)',
    fillcolor = 'rgba(233, 71, 74,.6)'
))

# mean line
#fig.add_hline(y=mean, annotation_text="{:.2f}".format(mean) + ' (overall workload)', annotation_position="right", line_color='red', opacity=.5)

# layout and axes
fig.update_traces(boxpoints='all')
fig.update_yaxes(title="workload (%)", range=[-3, 103])
fig.update_xaxes(tickprefix='Block ')
fig.update_layout(showlegend = False, plot_bgcolor = 'rgba(239, 240, 231,.7)')

pio.write_image(fig, 'NASA_TLX_workload.pdf')
fig.show()

In [91]:
# Interactive chart: filter by block using the legend on the right

# Box plots in Plotly Express, see: https://plotly.com/python/box-plots
# In a box plot created by px.box, the distribution of the column given as y argument is represented.
# If a column name is given as x argument, a box plot is drawn for each value of x.

# Columns: Index / Block / mental effort / physical effort / temporal effort / performance / effort / frustration
data = df_nasa_rtlx.iloc[:,1:8]

# Create per-block views to draw one box trace per block.
b1 = data[data['Block'] == 1]
b2 = data[data['Block'] == 2]
b3 = data[data['Block'] == 3]

fig = go.Figure()
fig.add_trace(go.Box(
    y=b1['mental effort'],
    x=b1['Block'],
    name='Block 1',
    marker_color='rgba(54, 161, 185,1)',
    fillcolor='rgba(54, 161, 185,.5)'
))

fig.add_trace(go.Box(
    y=b2['mental effort'],
    x=b2['Block'],
    name='Block 2',
    marker_color='rgba(0, 44, 61,.8)',
    fillcolor = 'rgba(0, 44, 61,.4)'
))

fig.add_trace(go.Box(
    y=b3['mental effort'],
    x=b3['Block'],
    name='Block 3',
    marker_color='rgba(233, 71, 74,.8)',
    fillcolor = 'rgba(233, 71, 74,.6)'
))

fig.update_traces(boxpoints='all')
fig.update_yaxes(title="mental effort (%)", range=[-3, 103])
fig.update_xaxes(tickprefix='Block ')

fig.update_layout(
    showlegend = False,
    plot_bgcolor = 'rgba(239, 240, 231,.7)'
)

pio.write_image(fig, 'NASA_TLX_mental_effort.pdf')
fig.show()

### Dimension-Level Analysis

In [92]:
df_nasa_rtlx_1.describe() # statistics for block 1

,PID,Block,mental effort,physical effort,temporal effort,performance,effort,frustration,overall workload
count,23.000000,23.0,23.000000,23.000000,23.000000,23.000000,23.000000,23.000000,23.000000
mean,12.608696,1.0,40.000000,41.521739,32.173913,45.869565,51.521739,48.913043,43.333333
std,7.209458,0.0,26.068093,20.639480,23.297780,23.676425,20.194215,27.918572,13.775802
min,1.000000,1.0,5.000000,0.000000,0.000000,0.000000,15.000000,0.000000,19.166667
25%,6.500000,1.0,17.500000,22.500000,17.500000,32.500000,35.000000,25.000000,34.166667
50%,13.000000,1.0,35.000000,40.000000,25.000000,45.000000,50.000000,50.000000,41.666667
75%,18.500000,1.0,65.000000,60.000000,50.000000,60.000000,67.500000,72.500000,49.583333
max,24.000000,1.0,85.000000,75.000000,80.000000,85.000000,90.000000,95.000000,73.333333


In [93]:
df_nasa_rtlx_1.iloc[:,2:].corr(method='pearson', min_periods=1)

,mental effort,physical effort,temporal effort,performance,effort,frustration,overall workload
mental effort,1.000000,-0.128837,0.230143,0.069964,0.403666,0.220158,0.541112
physical effort,-0.128837,1.000000,0.144054,0.236688,0.484946,-0.294784,0.336389
temporal effort,0.230143,0.144054,1.000000,0.604144,0.497452,0.030004,0.695152
performance,0.069964,0.236688,0.604144,1.000000,0.479577,0.168250,0.711907
effort,0.403666,0.484946,0.497452,0.479577,1.000000,0.154235,0.822411
frustration,0.220158,-0.294784,0.030004,0.168250,0.154235,1.000000,0.427933
overall workload,0.541112,0.336389,0.695152,0.711907,0.822411,0.427933,1.000000


In [94]:
df_nasa_rtlx_2.describe() # statistics for block 2

,PID,Block,mental effort,physical effort,temporal effort,performance,effort,frustration,overall workload
count,23.000000,23.0,23.000000,23.000000,23.000000,23.000000,23.000000,23.000000,23.000000
mean,12.608696,2.0,37.608696,41.521739,28.478261,43.913043,48.260870,48.913043,41.449275
std,7.209458,0.0,20.442245,22.835120,18.242980,23.882122,20.538694,24.259793,14.167151
min,1.000000,2.0,10.000000,5.000000,0.000000,0.000000,5.000000,0.000000,8.333333
25%,6.500000,2.0,22.500000,20.000000,20.000000,25.000000,37.500000,35.000000,30.000000
50%,13.000000,2.0,30.000000,45.000000,25.000000,45.000000,50.000000,55.000000,43.333333
75%,18.500000,2.0,55.000000,57.500000,35.000000,67.500000,62.500000,67.500000,50.000000
max,24.000000,2.0,80.000000,80.000000,70.000000,80.000000,80.000000,85.000000,65.833333


In [95]:
df_nasa_rtlx_3.describe() # statistics for block 3

,PID,Block,mental effort,physical effort,temporal effort,performance,effort,frustration,overall workload
count,23.000000,23.0,23.000000,23.000000,23.000000,23.000000,23.000000,23.000000,23.000000
mean,12.608696,3.0,34.565217,44.347826,31.086957,42.173913,47.608696,45.652174,40.905797
std,7.209458,0.0,27.090465,27.440197,22.205472,23.443650,25.489283,27.564154,19.721727
min,1.000000,3.0,5.000000,0.000000,0.000000,0.000000,5.000000,0.000000,9.166667
25%,6.500000,3.0,15.000000,25.000000,17.500000,27.500000,30.000000,22.500000,28.333333
50%,13.000000,3.0,25.000000,40.000000,25.000000,45.000000,45.000000,45.000000,35.000000
75%,18.500000,3.0,45.000000,67.500000,47.500000,55.000000,75.000000,67.500000,54.583333
max,24.000000,3.0,90.000000,90.000000,80.000000,85.000000,80.000000,90.000000,76.666667


In [96]:
correlations = df_nasa_rtlx_1.iloc[:,2:].corr(method='pearson', min_periods=1)
correlations

,mental effort,physical effort,temporal effort,performance,effort,frustration,overall workload
mental effort,1.000000,-0.128837,0.230143,0.069964,0.403666,0.220158,0.541112
physical effort,-0.128837,1.000000,0.144054,0.236688,0.484946,-0.294784,0.336389
temporal effort,0.230143,0.144054,1.000000,0.604144,0.497452,0.030004,0.695152
performance,0.069964,0.236688,0.604144,1.000000,0.479577,0.168250,0.711907
effort,0.403666,0.484946,0.497452,0.479577,1.000000,0.154235,0.822411
frustration,0.220158,-0.294784,0.030004,0.168250,0.154235,1.000000,0.427933
overall workload,0.541112,0.336389,0.695152,0.711907,0.822411,0.427933,1.000000


In [97]:
correlations = df_nasa_rtlx.iloc[:,2:].corr(method='pearson', min_periods=1)
ticklabels = ['me','pe','te','p','e','f']

fig = go.Figure(data=go.Heatmap(
                    colorscale=[(0, "#e21c36"), (0.5, "#eff0e7"), (1, "#002c3d")],
                    z=correlations,        
                    zmin=-1,
                    zmax=1,
                    x=ticklabels,
                    y=ticklabels,
                    hoverinfo='text',
                    hovertext=correlations,
                    hoverongaps = False))

fig.update_xaxes(dtick=1)
fig.update_layout(
    title="Pearson's correlation",
    #height = 800,
    annotations=[
            go.layout.Annotation(
                text='me / pe / te = mental / physical / temporal effort, p = performance, e = effort, f = frustration',
                align='left',
                showarrow=False,
                xref='paper',
                yref='paper',
                x=0,
                y=-0.15
            )
        ]
)

pio.write_image(fig, 'NASA_TLX_Pearson_all.pdf')
fig.show()

In [98]:
# calculate Pearson's correlation coefficients per block (revealing linear correlations in NASA-TLX dimensions)
correlations_1 = df_nasa_rtlx_1.iloc[:,2:].corr(method='pearson', min_periods=1)
correlations_2 = df_nasa_rtlx_2.iloc[:,2:].corr(method='pearson', min_periods=1)
correlations_3 = df_nasa_rtlx_3.iloc[:,2:].corr(method='pearson', min_periods=1)

# for shorter labels in the chart
ticklabels = ['me','pe','te','p','e','f']

# DRY
abc = dict(
        zmin=-1,
        zmax=1,
        x=ticklabels,
        y=ticklabels,
        colorscale=[(0, "#e21c36"), (0.5, "#eff0e7"), (1, "#002c3d")],
        hoverinfo='text')

# plots
fig = make_subplots(
        rows=1, cols=3,
        subplot_titles=('Block 1', 'Block 2', 'Block 3'))

fig.add_trace(
    go.Heatmap(
        abc,
        z=correlations_1,        
        hovertext=correlations_1),
    row=1, col=1)

fig.add_trace(
     go.Heatmap(
        abc,
        z=correlations_2,        
        hovertext=correlations_2),
     row=1, col=2)

fig.add_trace(
    go.Heatmap(
        abc,
        z=correlations_3,        
        hovertext=correlations_3),
     row=1, col=3)

fig.update_layout(
    title="Pearson's correlations",
    height = 400)

#fig.update_yaxes(autorange="reversed")

# legend below plots
fig.add_annotation(
    text="me / pe / te = mental / physical / temporal effort, p = performance, e = effort, f = frustration",
    xref="paper", yref="paper", x=0, y=-0.3, showarrow=False)

pio.write_image(fig, 'NASA_TLX_Person_blockwise.pdf')
fig.show()

In [99]:
# MENTAL EFFORT

# Box plots in Plotly Express, see: https://plotly.com/python/box-plots
# In a box plot created by px.box, the distribution of the column given as y argument is represented.
# If a column name is given as x argument, a box plot is drawn for each value of x.

data = df_nasa_rtlx.iloc[:,1:8]

b1 = data[data['Block'] == 1]
b2 = data[data['Block'] == 2]
b3 = data[data['Block'] == 3]

fig = go.Figure()
fig.add_trace(go.Box(
    y=b1['mental effort'],
    x=b1['Block'],
    name='Block 1',
    marker_color='rgba(54, 161, 185,1)',
    fillcolor='rgba(54, 161, 185,.5)'
))

fig.add_trace(go.Box(
    y=b2['mental effort'],
    x=b2['Block'],
    name='Block 2',
    marker_color='rgba(0, 44, 61,.8)',
    fillcolor = 'rgba(0, 44, 61,.4)'
))

fig.add_trace(go.Box(
    y=b3['mental effort'],
    x=b3['Block'],
    name='Block 3',
    marker_color='rgba(233, 71, 74,.8)',
    fillcolor = 'rgba(233, 71, 74,.6)'
))

fig.update_traces(boxpoints='all')
fig.update_yaxes(title="mental effort (%)", range=[-3, 103])
fig.update_xaxes(tickprefix='Block ')

fig.update_layout(
    showlegend = False,
    plot_bgcolor = 'rgba(239, 240, 231,.7)'
)

pio.write_image(fig, 'NASA_TLX_mental_effort.pdf')
fig.show()

In [100]:
# PHYSICAL EFFORT

data = df_nasa_rtlx.iloc[:,1:8]

b1 = data[data['Block'] == 1]
b2 = data[data['Block'] == 2]
b3 = data[data['Block'] == 3]

fig = go.Figure()
fig.add_trace(go.Box(
    y=b1['physical effort'],
    x=b1['Block'],
    name='Block 1',
    marker_color='rgba(54, 161, 185,1)',
    fillcolor='rgba(54, 161, 185,.5)'
))

fig.add_trace(go.Box(
    y=b2['physical effort'],
    x=b2['Block'],
    name='Block 2',
    marker_color='rgba(0, 44, 61,.8)',
    fillcolor = 'rgba(0, 44, 61,.4)'
))

fig.add_trace(go.Box(
    y=b3['physical effort'],
    x=b3['Block'],
    name='Block 3',
    marker_color='rgba(233, 71, 74,.8)',
    fillcolor = 'rgba(233, 71, 74,.6)'
))

fig.update_traces(boxpoints='all')
fig.update_yaxes(title="physical effort (%)", range=[-3, 103])
fig.update_xaxes(tickprefix='Block ')

fig.update_layout(
    showlegend = False,
    plot_bgcolor = 'rgba(239, 240, 231,.7)'
)

pio.write_image(fig, 'NASA_TLX_physical_effort.pdf')
fig.show()

In [101]:
# TEMPORAL EFFORT

data = df_nasa_rtlx.iloc[:,1:8]

b1 = data[data['Block'] == 1]
b2 = data[data['Block'] == 2]
b3 = data[data['Block'] == 3]

fig = go.Figure()
fig.add_trace(go.Box(
    y=b1['temporal effort'],
    x=b1['Block'],
    name='Block 1',
    marker_color='rgba(54, 161, 185,1)',
    fillcolor='rgba(54, 161, 185,.5)'
))

fig.add_trace(go.Box(
    y=b2['temporal effort'],
    x=b2['Block'],
    name='Block 2',
    marker_color='rgba(0, 44, 61,.8)',
    fillcolor = 'rgba(0, 44, 61,.4)'
))

fig.add_trace(go.Box(
    y=b3['temporal effort'],
    x=b3['Block'],
    name='Block 3',
    marker_color='rgba(233, 71, 74,.8)',
    fillcolor = 'rgba(233, 71, 74,.6)'
))

fig.update_traces(boxpoints='all')
fig.update_yaxes(title="temporal effort (%)", range=[-3, 103])
fig.update_xaxes(tickprefix='Block ')

fig.update_layout(
    showlegend = False,
    plot_bgcolor = 'rgba(239, 240, 231,.7)'
)

pio.write_image(fig, 'NASA_TLX_temporal_effort.pdf')
fig.show()

In [102]:
# PERFORMANCE

data = df_nasa_rtlx.iloc[:,1:8]

b1 = data[data['Block'] == 1]
b2 = data[data['Block'] == 2]
b3 = data[data['Block'] == 3]

fig = go.Figure()
fig.add_trace(go.Box(
    y=b1['performance'],
    x=b1['Block'],
    name='Block 1',
    marker_color='rgba(54, 161, 185,1)',
    fillcolor='rgba(54, 161, 185,.5)'
))

fig.add_trace(go.Box(
    y=b2['performance'],
    x=b2['Block'],
    name='Block 2',
    marker_color='rgba(0, 44, 61,.8)',
    fillcolor = 'rgba(0, 44, 61,.4)'
))

fig.add_trace(go.Box(
    y=b3['performance'],
    x=b3['Block'],
    name='Block 3',
    marker_color='rgba(233, 71, 74,.8)',
    fillcolor = 'rgba(233, 71, 74,.6)'
))

fig.update_traces(boxpoints='all')
fig.update_yaxes(title="performance (%)", range=[-3, 103])
fig.update_xaxes(tickprefix='Block ')

fig.update_layout(
    showlegend = False,
    plot_bgcolor = 'rgba(239, 240, 231,.7)'
)

pio.write_image(fig, 'NASA_TLX_performance.pdf')
fig.show()

In [103]:
# EFFORT

data = df_nasa_rtlx.iloc[:,1:8]

b1 = data[data['Block'] == 1]
b2 = data[data['Block'] == 2]
b3 = data[data['Block'] == 3]

fig = go.Figure()
fig.add_trace(go.Box(
    y=b1['effort'],
    x=b1['Block'],
    name='Block 1',
    marker_color='rgba(54, 161, 185,1)',
    fillcolor='rgba(54, 161, 185,.5)'
))

fig.add_trace(go.Box(
    y=b2['effort'],
    x=b2['Block'],
    name='Block 2',
    marker_color='rgba(0, 44, 61,.8)',
    fillcolor = 'rgba(0, 44, 61,.4)'
))

fig.add_trace(go.Box(
    y=b3['effort'],
    x=b3['Block'],
    name='Block 3',
    marker_color='rgba(233, 71, 74,.8)',
    fillcolor = 'rgba(233, 71, 74,.6)'
))

fig.update_traces(boxpoints='all')
fig.update_yaxes(title="effort (%)", range=[-3, 103])
fig.update_xaxes(tickprefix='Block ')

fig.update_layout(
    showlegend = False,
    plot_bgcolor = 'rgba(239, 240, 231,.7)'
)

pio.write_image(fig, 'NASA_TLX_effort.pdf')
fig.show()

In [104]:
# FRUSTRATION

data = df_nasa_rtlx[["Block","frustration"]]

b1 = data[data['Block'] == 1]
b2 = data[data['Block'] == 2]
b3 = data[data['Block'] == 3]

fig = go.Figure()
fig.add_trace(go.Box(
    y=b1['frustration'],
    x=b1['Block'],
    name='Block 1',
    marker_color='rgba(54, 161, 185,1)',
    fillcolor='rgba(54, 161, 185,.5)'
))

fig.add_trace(go.Box(
    y=b2['frustration'],
    x=b2['Block'],
    name='Block 2',
    marker_color='rgba(0, 44, 61,.8)',
    fillcolor = 'rgba(0, 44, 61,.4)'    
))

fig.add_trace(go.Box(
    y=b3['frustration'],
    x=b3['Block'],
    name='Block 3',
    marker_color='rgba(233, 71, 74,.8)',
    fillcolor = 'rgba(233, 71, 74,.6)'
))

fig.update_traces(boxpoints='all')
fig.update_yaxes(title="frustration (%)", range=[-3, 103])
fig.update_xaxes(tickprefix='Block ')

fig.update_layout(
    showlegend = False,
    plot_bgcolor = 'rgba(239, 240, 231,.7)'
)

pio.write_image(fig, 'NASA_TLX_frustration.pdf')
fig.show()

## User Experience

In [108]:
df_ux = pd.read_csv(rf'{data_folder}UX.tsv', sep='\t') # alle UX Daten
df_ux.head(2)

,Zeitstempel,Probanden-ID,Unnamed: 2,Unnamed: 3,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7,Unnamed: 8,Unnamed: 9,...,Unnamed: 20,Unnamed: 21,Unnamed: 22,Unnamed: 23,Unnamed: 24,Unnamed: 25,Unnamed: 26,Unnamed: 27,Unnamed: 28,Unnamed: 29
0,10.05.2021 15:06:42,1,4,6,4,6,6,7,7,5,...,6,3,6,7,5,4,5,7,6,6
1,11.05.2021 15:55:53,2,3,7,5,6,6,6,5,6,...,7,5,7,5,6,6,6,6,7,6


In [ ]:
df_ux.describe()

,Probanden-ID,Unnamed: 2,Unnamed: 3,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7,Unnamed: 8,Unnamed: 9,Unnamed: 10,...,Unnamed: 20,Unnamed: 21,Unnamed: 22,Unnamed: 23,Unnamed: 24,Unnamed: 25,Unnamed: 26,Unnamed: 27,Unnamed: 28,Unnamed: 29
count,24.000000,24.000000,24.000000,24.000000,24.000000,24.000000,24.000000,24.000000,24.000000,24.00000,...,24.0000,24.000000,24.000000,24.000000,24.000000,24.000000,24.000000,24.000000,24.000000,24.000000
mean,12.500000,3.625000,4.875000,3.875000,4.166667,3.875000,4.666667,4.083333,4.833333,4.00000,...,4.8750,5.083333,5.791667,4.875000,4.833333,5.125000,5.125000,5.250000,5.125000,4.250000
std,7.071068,1.244553,1.190999,1.034723,1.434563,1.776966,1.493949,1.442120,1.203859,1.47442,...,1.2619,0.974308,0.832971,1.034723,0.868115,0.991814,1.153916,0.944089,0.946963,1.224745
min,1.000000,1.000000,2.000000,2.000000,2.000000,1.000000,1.000000,2.000000,2.000000,1.00000,...,2.0000,3.000000,4.000000,3.000000,3.000000,2.000000,2.000000,3.000000,3.000000,2.000000
25%,6.750000,3.000000,4.000000,3.000000,3.000000,2.000000,4.000000,3.000000,4.000000,3.00000,...,4.0000,5.000000,5.000000,4.000000,4.000000,5.000000,4.750000,5.000000,5.000000,3.000000
50%,12.500000,3.000000,5.000000,4.000000,4.000000,3.500000,5.000000,3.500000,5.000000,4.00000,...,5.0000,5.000000,6.000000,5.000000,5.000000,5.000000,5.000000,5.000000,5.000000,4.000000
75%,18.250000,4.250000,5.250000,5.000000,5.250000,6.000000,6.000000,5.000000,6.000000,5.00000,...,6.0000,6.000000,6.000000,6.000000,5.000000,6.000000,6.000000,6.000000,6.000000,5.000000
max,24.000000,6.000000,7.000000,6.000000,7.000000,6.000000,7.000000,7.000000,6.000000,6.00000,...,7.0000,7.000000,7.000000,7.000000,7.000000,6.000000,7.000000,7.000000,7.000000,6.000000


In [ ]:
# normalize the values 
np_arr = df_ux.iloc[:,2:].values # returns a numpy array
scaler = preprocessing.MinMaxScaler(feature_range=(-1, 1)) # map [1,7] to [-1,1]
np_arr_scaled = scaler.fit_transform(np_arr)
normalized_values = pd.DataFrame(np_arr_scaled)
normalized_values

,0,1,2,3,4,5,6,7,8,9,...,18,19,20,21,22,23,24,25,26,27
0,0.2,0.6,0.0,0.6,1.0,1.000000,1.0,0.5,-1.0,-0.2,...,0.6,-1.0,0.333333,1.0,0.0,0.0,0.2,1.0,0.5,1.0
1,-0.2,1.0,0.5,0.6,1.0,0.666667,0.2,1.0,0.2,1.0,...,1.0,0.0,1.000000,0.0,0.5,1.0,0.6,0.5,1.0,1.0
2,1.0,-0.6,-0.5,-0.2,-0.6,-1.000000,-1.0,1.0,0.6,-0.2,...,-0.6,-0.5,0.333333,0.5,-0.5,1.0,0.6,0.0,0.5,0.0
3,0.6,0.2,-0.5,-0.6,-0.2,0.333333,0.2,0.0,0.6,0.6,...,0.6,1.0,0.333333,0.5,0.5,1.0,0.6,0.5,0.0,0.5
4,-0.2,0.2,-0.5,0.6,-0.2,0.666667,-0.6,-0.5,0.6,0.2,...,0.2,0.0,-0.333333,-0.5,-0.5,0.5,-0.2,0.0,0.0,0.0
5,0.2,0.2,0.5,-0.2,0.6,0.666667,0.2,1.0,0.6,0.2,...,0.2,0.0,0.333333,0.0,0.0,0.5,0.2,0.0,0.0,0.0
6,-0.6,0.2,-0.5,-0.6,-0.6,0.666667,-0.6,0.5,-0.6,1.0,...,1.0,-0.5,0.333333,0.0,0.0,1.0,-0.2,0.0,0.0,-0.5
7,0.6,0.6,0.0,-0.6,-0.6,0.000000,-0.6,1.0,-1.0,-1.0,...,-1.0,0.0,-1.000000,-0.5,0.0,1.0,0.6,0.0,0.5,-0.5
8,-0.2,0.2,0.0,-0.2,-0.2,0.000000,0.6,0.5,-0.2,1.0,...,0.6,0.5,0.333333,0.5,0.0,0.5,0.6,0.5,0.0,0.0
9,0.2,-0.6,-0.5,0.2,1.0,0.000000,0.2,-0.5,0.6,0.6,...,-0.6,0.5,-1.000000,0.0,0.0,0.5,0.2,-0.5,-0.5,-0.5


In [ ]:
# Load the German word-pair definitions used in this analysis.
df_wortpaare = pd.read_csv('Wortpaare.tsv', sep='\t', header=None)

# Harmonize column names for plotting and profile grouping.
df_wortpaare.rename(columns={df_wortpaare.columns[0]:'neg', df_wortpaare.columns[1]: 'pos', df_wortpaare.columns[2]: 'AttrakDiff-Profil'}, inplace=True)

# Aggregate normalized ratings across participants for each word pair.
df_wortpaare['Rating'] = normalized_values.mean().apply("{0:.3f}".format)

# Convert to numeric values so sorting and plotting work reliably.
df_wortpaare['Rating'] = pd.to_numeric(df_wortpaare['Rating'])

# Sort from most positive to most negative and reset the display index.
df_wortpaare.sort_values(by=['Rating'], ascending=False, inplace=True)
df_wortpaare.reset_index(drop=True, inplace=True)

df_wortpaare

,neg,pos,AttrakDiff-Profil,Rating
0,unsympathisch,sympatisch,ATT,0.562
1,konventionell,originell,HQ-S,0.517
2,stillos,stilvoll,HQ-I,0.417
3,isolierend,verbindend,HQ-I,0.417
4,phantasielos,kreativ,HQ-S,0.333
5,minderwertig,wertvoll,HQ-I,0.278
6,zurückweisend,einladend,ATT,0.250
7,verwirrend,übersichtlich,PQ,0.222
8,nicht vorzeigbar,vorzeigbar,HQ-I,0.208
9,laienhaft,fachmännisch,HQ-I,0.200


In [ ]:
# Load the English word-pair definitions for publication-ready charts.
df_word_pairs = pd.read_csv('Word-Pairs.tsv', sep='\t', header=None)

# Harmonize column names with the profile-based analysis.
df_word_pairs.rename(columns={df_word_pairs.columns[0]:'neg', df_word_pairs.columns[1]:'pos', df_word_pairs.columns[2]:'AttrakDiff-Profile'}, inplace=True)

# Reuse the participant-averaged normalized ratings from the UX preprocessing step.
df_word_pairs['Rating'] = normalized_values.mean().apply("{0:.3f}".format)

# Convert to numeric values and order profiles by rating strength.
df_word_pairs['Rating'] = pd.to_numeric(df_word_pairs['Rating'])
df_word_pairs.sort_values(by=['Rating'], ascending=False, inplace=True)
df_word_pairs.reset_index(drop=True, inplace=True)

#df_wortpaare.sample(3)
df_word_pairs

,neg,pos,AttrakDiff-Profile,Rating
0,disagreeable,likeable,ATT,0.562
1,conventional,inventive,HQ-S,0.517
2,tacky,stylish,HQ-I,0.417
3,isolating,connective,HQ-I,0.417
4,unimaginative,creative,HQ-S,0.333
5,cheap,premium,HQ-I,0.278
6,rejecting,inviting,ATT,0.250
7,confusing,clearly structured,PQ,0.222
8,unpresentable,presentable,HQ-I,0.208
9,unprofessional,professional,HQ-I,0.200


In [ ]:
means = df_word_pairs.groupby('AttrakDiff-Profile').mean() 
df_word_pairs.groupby('AttrakDiff-Profile').describe()

Rating                                                    \
                    count      mean       std    min     25%    50%     75%   
AttrakDiff-Profile                                                            
ATT                   7.0  0.139857  0.218804 -0.083  0.0000  0.125  0.1875   
HQ-I                  7.0  0.190429  0.212990 -0.104  0.0585  0.208  0.3475   
HQ-S                  7.0  0.222286  0.156143  0.042  0.1445  0.181  0.2635   
PQ                    7.0  0.030000  0.152701 -0.167 -0.0975  0.050  0.1500   

                           
                      max  
AttrakDiff-Profile         
ATT                 0.562  
HQ-I                0.417  
HQ-S                0.517  
PQ                  0.222

In [ ]:
data = df_wortpaare
# texts = np.where(df_wortpaare['Rating'] < 0, df_wortpaare['neg'], df_wortpaare['pos'])
colors = np.where(df_wortpaare['Rating'] < 0, 'red', 'blue')

fig = px.scatter(data, 
    x="AttrakDiff-Profil", 
    y="Rating", 
    # text=texts, 
    color=colors, 
    marginal_y="histogram",
    hover_data=["neg", "pos"],
    opacity=0.8
)

# fig.update_traces(marker_size=10)
# fig.update_traces(textposition='top center')
# fig.update_yaxes(range=[-1, 1])

fig.update_layout(
    showlegend = False,
    plot_bgcolor = 'rgba(239, 240, 231,.7)'
)

pio.write_image(fig, 'AttrakDiff_Histogram.pdf')
fig.show()

In [ ]:
# ATTEMPT 2
data = df_wortpaare

fig = make_subplots(
    rows=1, cols=2, 
    shared_yaxes=True, shared_xaxes=False,
    column_widths=[0.85, 0.25],
    horizontal_spacing=0.01
)

fig.add_trace(
    go.Scatter(
        y = data['Rating'],
        x = data['AttrakDiff-Profil'],
        mode='markers',
        marker=dict(
            size=10,
            color=np.where(data['Rating'] < 0, '#E21C36', '#47626E'),
            opacity=0.5
        )
    ), row=1, col=1
)

# fig.add_trace(
#     go.Histogram( 
#         y=data['Rating'], name='y density', marker=dict(color='#36A1B9', opacity=.5)
#     ), row=1, col=2
# )

fig.update_layout(
    showlegend = False,
    plot_bgcolor = 'rgba(239, 240, 231,.7)',
    width=600,
    height=500
)

pio.write_image(fig, 'AttrakDiff_transparent.pdf')
fig.show()

In [ ]:
# ATTEMPT 2
data = df_wortpaare

d1= data[data['Rating'] < 0]
d2 = data[data['Rating'] >= 0]

fig = go.Figure()

fig.add_trace(
    go.Box(
        y = d1['Rating'],
        x = d1['AttrakDiff-Profil'],
        marker_color='#E21C36',
        fillcolor = 'rgba(0,0,0,0)',
        line_color='rgba(0,0,0,0)',
        marker_size=12,
        #opacity=.8
    )
)

fig.add_trace(
    go.Box(
        y = d2['Rating'],
        x = d2['AttrakDiff-Profil'],
        marker_color='#47626E',
        fillcolor = 'rgba(0,0,0,0)',
        line_color='rgba(0,0,0,0)',
        marker_size=12,
        #opacity=.8
    )
)

fig.update_traces(
    boxpoints='all',
    jitter=0.8,
    pointpos=0
)

fig.update_layout(    
    font_family='Roboto Condensed',
    font_size=18,
    height=350,
    margin=go.layout.Margin(
        l=0, #left margin
        r=0, #right margin
        b=0, #bottom margin
        t=0, #top margin
    ),
    plot_bgcolor = 'rgba(239, 240, 231,.7)',
    showlegend = False
)

pio.write_image(fig, 'AttrakDiff.pdf')
fig.show()

In [ ]:
data = df_wortpaare

fig = px.strip(
        data,
        y = data['Rating'],
        x = data['AttrakDiff-Profil'],
        color=np.where(data['Rating'] < 0, '#E21C36', '#47626E')
)

fig.update_layout(
    showlegend = False,
    plot_bgcolor = 'rgba(239, 240, 231,.7)',
    width=800,
    height=500,
    font_size=24
)

fig.show()

In [ ]:
data = df_word_pairs

fig = go.FigureWidget()

# HQ
fig.add_trace(go.Violin(y=data['Rating'][ data['AttrakDiff-Profile'].str.startswith('HQ') ],
                        scalegroup='profil', name='HQ',
                        line_color='#47626E')
             )

# PQ
fig.add_trace(go.Violin(y=data['Rating'][ data['AttrakDiff-Profile'] == 'PQ' ],
                        scalegroup='profil', name='PQ',
                        line_color='#47626E')
             )

fig.update_traces(meanline_visible=True,
                  points='all', # show all points
                  jitter=0.5,  # for better visibility
                  scalemode='count') # scale with total count

fig.update_layout(
   # title_text="<b>AttrakDiff dimensions</b>",
    violingap=0, violingroupgap=0.4, violinmode='overlay',
    showlegend=False,
    yaxis1=dict(
        title='Rating',
        range = [-1,1]
    ),
    plot_bgcolor = 'rgba(239, 240, 231,.7)'
)

pio.write_image(fig, 'AttrakDiff_HQ_PQ.pdf')
fig.show()

Source: Hassenzahl, Marc et al. “AttrakDiff: A questionnaire for measuring perceived hedonic and pragmatic quality.” Mensch & Computer (2003). DOI: 10.1007/978-3-322-80058-9_19.

**Pragmatic Quality (PQ)**   
If an interactive product is suitable for manipulating the environment and is perceived that way by its users, it has “pragmatic” quality. Common principles, rules, and methods from usability engineering and software ergonomics often emphasize pragmatic quality in a one-sided way.

**Hedonic Quality / Stimulation (HQ-S)**  
People strive for personal growth, i.e., improving their knowledge and skills. Products can support this development by being stimulating. Novel, interesting, and inspiring functionalities, content, interaction styles, and presentation styles can increase attention, reduce motivational problems, and make it easier to find new solutions for existing problems. In this way, stimulation can also indirectly support task completion.

**Hedonic Quality / Identity (HQ-I)**   
People also express their self through objects. They want to be perceived by relevant others in a specific way. A product can support this by communicating a desired identity.

*Different product characters can emerge from the combination of hedonic and pragmatic qualities. From our perspective, a product is desirable when both qualities are strongly expressed.*

**Attractiveness (ATT)**   
We distinguish perceived pragmatic and hedonic quality from a product's attractiveness. The attractiveness judgment (“good,” “likable,” “motivating”) is a global evaluation based on perceived qualities. It is assumed that perceiving a product as pragmatic or hedonic remains relatively stable across situations, while the global evaluation can still change.

In [ ]:
data = df_word_pairs

fig = go.Figure()

# ATT
fig.add_trace(go.Violin(y=data['Rating'][ data['AttrakDiff-Profile'] == 'ATT' ],
                        scalegroup='profil', name='ATT',
                        line_color='#47626E')
             )
# HQ-S
fig.add_trace(go.Violin(y=data['Rating'][ data['AttrakDiff-Profile'] == 'HQ-S' ],
                        scalegroup='profil', name='HQ-S',
                        line_color='#47626E')
             )

# HQ-I
fig.add_trace(go.Violin(y=data['Rating'][ data['AttrakDiff-Profile'] == 'HQ-I' ],
                        scalegroup='profil', name='HQ-I',
                        line_color='#47626E')
             )

# PQ
fig.add_trace(go.Violin(y=data['Rating'][ data['AttrakDiff-Profile'] == 'PQ' ],
                        scalegroup='profil', name='PQ',
                        line_color='#47626E')
             )

fig.update_traces(meanline_visible=True,
                  points='all', # show all points
                  jitter=0.5,  # for better visibility
                  scalemode='count') # scale with total count

fig.update_layout(
    #title_text="<b>AttrakDiff dimensions</b>",
    violingap=0, violingroupgap=0.4, violinmode='overlay',
    showlegend=False,
    yaxis1=dict(
        title='Rating',
        range = [-1,1]
    ),
    plot_bgcolor = 'rgba(239, 240, 231,.7)'
)

pio.write_image(fig, 'AttrakDiff_Profiles.pdf')
fig.show()

In [ ]:
# from plotly.subplots import make_subplots # Needed to create figure with two y-axis for labelling

fig = make_subplots(specs=[[{"secondary_y": True}]])

colors = np.where(df_word_pairs['Rating'] < 0, '#E21C36', '#47626E')

# Add traces
fig.add_trace(
    go.Scatter(x=df_word_pairs['Rating'], y=df_word_pairs['neg'] + '&nbsp;', mode = 'markers', 
               marker_color = colors),
    secondary_y=False,
)

fig.add_trace(
    go.Scatter(x=df_word_pairs['Rating'], y='&nbsp;' + df_word_pairs['pos']+'&nbsp;' + '<b>(' + df_word_pairs['AttrakDiff-Profile'] + ')</b>', mode = 'markers', marker_size=10, marker_color = colors),
    secondary_y=True,
)

# Draw lines      
for i in range(0, 21):
               fig.add_shape(type='line',
                              x0 = 0, y0 = i,
                              x1 = df_word_pairs["Rating"][i],
                              y1 = i,
                              line=dict(color = '#47626E', width = 2))
for i in range(21, 28):
               fig.add_shape(type='line',
                              x0 = 0, y0 = i,
                              x1 = df_word_pairs["Rating"][i],
                              y1 = i,
                              line=dict(color = '#E21C36', width = 2))

range_val = df_word_pairs['Rating'].max()

fig.update_layout(
    xaxis = dict(
        dtick = 1,
        range = [-1,1]
    ),
    yaxis = dict(
        dtick=1,
        showgrid=False,
        autorange="reversed"
    ),
    yaxis2 = dict(
        dtick=1,
        autorange="reversed"
    ),
    font_family='Roboto Condensed',
    font_size=18,
    height = 850,
    margin=go.layout.Margin(
        l=0, #left margin
        r=0, #right margin
        b=0, #bottom margin
        t=0, #top margin
    ),
    plot_bgcolor = 'rgba(239, 240, 231,.7)',
    showlegend = False
)
 
pio.write_image(fig, 'wordpairs_profiles.pdf')
fig.show()

In [ ]:
# load data
df_log_mc = pd.read_csv('mapping_and_completion.tsv', sep='\t') 
df_log_mc

,mapping method,completion rate
0,condensed,"0,75"
1,condensed,"0,775"
2,condensed,"0,725"
3,equidistant,"0,735"
4,equidistant,"0,755"
5,equidistant,"0,71"
6,widening,"0,64"
7,widening,"0,61"
8,widening,"0,595"


In [ ]:
# see Plotly Error Bar Documentation: https://plotly.com/python/error-bars/

fig = go.Figure()
fig.add_trace(go.Bar(
    width=0.5,
    x=['condensed', 'equidistant', 'widening'], y=[0.75, 0.735, 0.62],
    error_y=dict(type='data', array=[0.025, 0.025, 0.025], width=15)
))

fig.add_hline(y=0.7, 
              line_color='rgb(233, 71, 74)', 
              annotation_text='<span style="color: rgb(233, 71, 74);">M=<br>0.7<span>',
              annotation_position='right',
              annotation_align='left')
             
fig.update_xaxes(title='layer arrangement')
fig.update_yaxes(title='completion rate',range=[0,1])

fig.update_layout(    
    font_family='Roboto Condensed',
    font_size=18,
    height=300,
    margin=go.layout.Margin(
        l=0, #left margin
        r=40, #right margin
        b=0, #bottom margin
        t=0, #top margin
    ),
    plot_bgcolor = 'rgba(239, 240, 231,.7)'
)
fig.update_traces(marker_color='rgba(0, 44, 61,.4)')


pio.write_image(fig, 'log_arrangement_and_completion.pdf')
fig.show()

In [ ]:
# see Plotly Error Bar Documentation: https://plotly.com/python/error-bars/

fig = go.Figure()
fig.add_trace(go.Bar(
    width=1.5,
    x=[6,9,12,15,18,21], y=[34,42.5,42.5,49,54,57],
    error_y=dict(type='data', array=[4,4,4,4,4,4], width=15)
))

fig.add_hline(y=47, 
              line_color='rgb(233, 71, 74)', 
              annotation_text='<span style="color: rgb(233, 71, 74);">M=<br>47<span>',
              annotation_position='right',
              annotation_align='left')
             
fig.update_xaxes(title='number of layers', dtick = 3)
fig.update_yaxes(title='trial duration (s)',range=[0,80])

fig.update_layout(    
    font_family='Roboto Condensed',
    font_size=18,
    height=300,
    margin=go.layout.Margin(
        l=0, #left margin
        r=0, #right margin
        b=0, #bottom margin
        t=0, #top margin
    ),
    plot_bgcolor = 'rgba(239, 240, 231,.7)'
)
fig.update_traces(marker_color='rgba(0, 44, 61,.4)')


pio.write_image(fig, 'log_layers_and_trials.pdf')
fig.show()

In [ ]:
# see Plotly Error Bar Documentation: https://plotly.com/python/error-bars/

fig = go.Figure()
fig.add_trace(go.Bar(
    width=1.5,
    x=[6,9,12,15,18,21], y=[1.825,2.2,2.7,3.275,3.575,3.875],
    error_y=dict(type='data', array=[0.25,0.25,0.25,0.25,0.25,0.25], width=15)
))

fig.add_hline(y=2.9, 
              line_color='rgb(233, 71, 74)', 
             # annotation_text='<span style="color: rgb(233, 71, 74);">M=<br>47<span>',
             # annotation_position='right',
             # annotation_align='left'
)
             
fig.update_xaxes(title='number of layers', dtick = 3)
fig.update_yaxes(title='number of turning points',range=[0,5])

fig.update_layout(    
    font_family='Roboto Condensed',
    font_size=18,
    height=400,
    margin=go.layout.Margin(
        l=0, #left margin
        r=0, #right margin > increase to show annotation text!
        b=0, #bottom margin
        t=0, #top margin
    ),
    plot_bgcolor = 'rgba(239, 240, 231,.7)'
)
fig.update_traces(marker_color='rgba(0, 44, 61,.4)')


pio.write_image(fig, 'log_layers_and_turning_points.pdf')
fig.show()

In [ ]:
# see Plotly Error Bar Documentation: https://plotly.com/python/error-bars/

fig = go.Figure()
fig.add_trace(go.Bar(
    width=1.5,
    x=[6,9,12,15,18,21], y=[0.37,0.435,0.47,0.495,0.485,0.485],
    error_y=dict(type='data', array=[0.05,0.05,0.05,0.05,0.05,0.05,], width=15)
))

fig.add_hline(y=0.46, 
              line_color='rgb(233, 71, 74)', 
             # annotation_text='<span style="color: rgb(233, 71, 74);">M=<br>47<span>',
             # annotation_position='right',
             # annotation_align='left'
)
             
fig.update_xaxes(title='number of layers', dtick = 3)
fig.update_yaxes(title='task duration (s)',range=[0,0.6])

fig.update_layout(    
    font_family='Roboto Condensed',
    font_size=18,
    height=300,
    margin=go.layout.Margin(
        l=0, #left margin
        r=0, #right margin > increase to show annotation text!
        b=0, #bottom margin
        t=0, #top margin
    ),
    plot_bgcolor = 'rgba(239, 240, 231,.7)'
)
fig.update_traces(marker_color='rgba(0, 44, 61,.4)')


pio.write_image(fig, 'log_layers_and_tasks.pdf')
fig.show()

In [ ]:
fig = make_subplots(rows=1, cols=3, subplot_titles=("trial duration (s)", "turning points", "task duration (s)"))

# subplot 1: trial duration - no. of layers
fig.add_trace(go.Bar(
        x=[6,9,12,15,18,21], y=[34,42.5,42.5,49,54,57],
        error_y=dict(type='data', array=[4,4,4,4,4,4], width=15)
    ), row=1, col=1)
fig.add_hline(y=47, line_color='rgb(233, 71, 74)', row=1, col=1)        
fig.update_xaxes(title=None, dtick = 3, row=1, col=1)
#fig.update_yaxes(title='trial duration (s)',range=[0,80], row=1, col=1)
fig.update_yaxes(title=None,range=[0,80], row=1, col=1)

# subplot 2: no. of turning points - no. of layers
fig.add_trace(go.Bar(
        x=[6,9,12,15,18,21], y=[1.825,2.2,2.7,3.275,3.575,3.875],
        error_y=dict(type='data', array=[0.25,0.25,0.25,0.25,0.25,0.25], width=15)
        ), row=1, col=2)
fig.add_hline(y=2.9, line_color='rgb(233, 71, 74)', row=1, col=2)        
fig.update_xaxes(title='no. of layers', title_font_family="Roboto Condensed", dtick = 3, row=1, col=2)
#fig.update_yaxes(title='no. of turning points',range=[0,5], row=1, col=2)
fig.update_yaxes(title=None,range=[0,5], row=1, col=2)

# subplot 3: task duration - no.of layers
fig.add_trace(go.Bar(
        x=[6,9,12,15,18,21], y=[0.37,0.435,0.47,0.495,0.485,0.485],
    error_y=dict(type='data', array=[0.05,0.05,0.05,0.05,0.05,0.05,], width=15)
    ), row=1, col=3)
fig.add_hline(y=0.46, line_color='rgb(233, 71, 74)', row=1, col=3)        
fig.update_xaxes(title=None, dtick = 3, row=1, col=3)
#fig.update_yaxes(title='task duration (s)',range=[0,0.6], row=1, col=3)
fig.update_yaxes(title=None,range=[0,0.6], row=1, col=3)

# update layout properties
fig.update_layout(    
    font_family='Roboto Condensed',
    font_size=18,
    height=350,
    margin=go.layout.Margin(
        l=0, #left margin
        r=0, #right margin > increase to show annotation text!
        b=0, #bottom margin
        t=40, #top margin
    ),
    plot_bgcolor = 'rgba(239, 240, 231,.7)',
    showlegend=False
)
fig.update_annotations(font_size=24)
fig.update_traces(marker_color='rgba(0, 44, 61,.4)')

pio.write_image(fig, 'subplots_no_of_layers.pdf')
fig.show()

# Observation Log

Data collection
- Non-participant observation
- Simultaneous *think-aloud* protocol
- Observed aspects included, for example: gestures used, relevant statements/reactions, body language, mood, and participant strategy when solving the assigned task

In [ ]:
# load data
df_beob = pd.read_csv('Beobachtungsprotokoll.tsv', sep='\t')

In [ ]:
# Rename columns for a consistent analysis schema.
df_beob.rename(columns={"Zeitstempel": "Timestamp", 
                            "Probanden-ID": "PID",
                            "Haben Sie allgemeine Anmerkungen zum Demonstrator?": "Anmerkungen",
                            "Wo sehen Sie dessen Stärken?": "Stärken",
                            "Wo sehen Sie dessen Schwächen?": "Schwächen",
                            "Was ist Ihr höchster Bildungsabschluss?": "Abschluss"},
                    inplace=True) 

# 1) Parse the start and end timestamps as datetime objects.
df_beob['Versuchsbeginn'] = pd.to_datetime(df_beob['Versuchsbeginn']) #.dt.strftime('%H:%M')
df_beob['Versuchsende'] = pd.to_datetime(df_beob['Versuchsende']) #.dt.strftime('%H:%M')
df_beob['Dauer'] = (df_beob['Versuchsende'] - df_beob['Versuchsbeginn'])

# 2) Ensure the computed duration is a proper timedelta.
df_beob['Dauer'] = pd.to_timedelta(df_beob['Dauer'])

# 3) Convert duration to whole minutes for descriptive statistics and charts.
df_beob['Minuten'] = df_beob['Dauer'].dt.total_seconds().div(60).astype(int)

# Show the first two rows as a quick data integrity check.
df_beob.head(2) # remove .head() to see all

,Timestamp,PID,Datum,Versuchsbeginn,Beobachtungsprotokoll,Anmerkungen,Stärken,Schwächen,Versuchsende,Dauer,Minuten
0,10.05.2021 15:11:50,1,10.05.2021,2022-02-02 14:10:00,Block 1: - geht vorsichtig vor - nutzt die gan...,"- kommt so vor, als würde es besser reagieren ...",- bei 3D-Zeichnungen (müsste viel genauer sein...,"- verhält sich meist vorhersehbar, aber manchm...",2022-02-02 15:11:00,0 days 01:01:00,61
1,11.05.2021 16:01:29,2,11.05.2021,2022-02-02 14:23:00,Block 1: - nutzt Zeigefinger oder Zeigefinger ...,- für zukünftige Tests: bisschen Variation wär...,"- Tiefenebenen nutzen, um Prozesse zu visualis...",- Anstrengung vom Drücken auf Dauer (besonders...,2022-02-02 16:01:00,0 days 01:38:00,98


### Statistical Analysis

In [ ]:
# mean, median, std,  etc. 
df_beob['Minuten'].describe().apply("{0:.2f}".format)

count     24.00
mean      80.54
std       15.82
min       53.00
25%       70.00
50%       77.50
75%       90.75
max      120.00
Name: Minuten, dtype: object

In [ ]:
# Calculate mean and standard deviation once for reusable chart overlays.
mean = df_beob['Minuten'].mean()
st_dev = df_beob['Minuten'].std()

# Bar chart
fig = go.Figure([go.Bar(x=df_beob['PID'], 
                        y=df_beob['Minuten'],
                        text=df_beob['Minuten'],
                        textposition='auto'
                )])
# mean line
fig.add_hline(y=mean, annotation_text="{:.2f}".format(mean) + ' (mean)', 
              annotation_position="right", line_color='red', opacity=.5)

# standard deviation rect
fig.add_hrect(y0=mean-st_dev, y1=mean+st_dev, line_width=0, fillcolor="red", opacity=0.1, annotation_text="Standard Deviation", 
              annotation_position="top right")

fig.update_layout(
    xaxis1=dict(
        title='PID',
        dtick=1 # show all ticks
    ),
    yaxis1=dict(
        title='Minutes',
        nticks=20
    ),    
    showlegend=False,
    title="Versuchsdauer in Minuten nach Proband*innen"
) 

fig.show()

**Key Facts**
- Average trial duration: 80.54 minutes
- Standard deviation: +/- 15.82 minutes
- Median: 77.50 minutes

**Discussion**

- Later sessions may also be shorter because the facilitators had gained more experience by that point.
- Stratifying by facilitator, participant gender, or participant prior experience could provide additional insights.

### Qualitative Analysis

Labeling of observation logs:

https://docs.google.com/document/d/10nXVY0wCIX8wB0ondRFI3g55qiVyPXF5JIT_0qA5SCM/edit?usp=sharing

- ``FIRST ACTION / REACTION``   
Uninfluenced spontaneous first expression.
- ``REACTIONS / STRATEGY``  
Assessment of perceived system properties, reactions to system changes/outputs, and the strategic approach used to construct a mental model of the system.
- ``PROBLEMS``  
Usability problems in operating the system (interaction impaired or impossible) caused by either (1) the programming/design of the application or (2) physical properties of the system / participant.
- ``GESTURES``   
Gestures used, including (if applicable) reason / intention / context.
- ``MOOD, ATTENTION, AND INTEREST``  
Current mood, invested mental capacity, and general attitude toward the system.
- ``MISCELLANEOUS``  
Best-of comments, suggestions for possible Elastic Display application areas, etc.

**Key Facts**
- ...
- ...

**Discussion**

- ...
- ...

## Considerations for Further Analysis

- H1: Height correlates with physical effort
- H2: Effort and frustration correlate negatively with performance
- H3: Participant gender affects self-assessment
- H4: Prior display experience and industry correlate positively with performance and negatively with effort and frustration
- additional hypotheses?

## Chart Scratchpad

In [ ]:
# Trial duration x PID (mean, median, standard deviation)

fig = make_subplots(rows=1, cols=2, 
                    column_widths=[0.85, 0.15],
                    subplot_titles=[
                        'Duration and Mean', 
                        'Median ± SD'
                    ],)
# Bar chart
fig.add_trace(go.Bar(x=df_beob['PID'], y=df_beob['Minuten'], text=df_beob['Minuten'], textposition='auto',), 
              row=1, col=1)

# Box plot
fig.add_trace(go.Box(y=df_beob['Minuten']),
              row=1, col=2)

fig.update_layout(
    xaxis1=dict(
        title='PID',
        dtick=1),
    yaxis1=dict(
        title='Minutes',
        nticks=20),
    xaxis2=dict(
        showticklabels=False),
    yaxis2=dict(
        nticks=20,
        rangemode="tozero"
    ),
    
    showlegend=False
)    

fig.show()

In [ ]:
# Trial duration (median, standard deviation)

data = df_beob[['PID','Minuten']]
fig = px.box(data,y='Minuten', points="all")

fig.show()